# dsPIC33AK256MC205 Firmware 


Firmware on the dsPIC33AK256MC205 used to generates raw data streams for every sensor in the ICS **random data generated currently** 

| Section | Sensor | Returns | Rate |
|---|---|---|---|
| 4 | LM95172 | `temp_cjc`, `temp_hjc` | 1 Hz |
| 5 | ADXRS645HDYZ | `omega_dps`, `rpm` | 100 Hz |
| 6 | ADXL206HDZ | `inclination`, `inclination_valid` | 1 Hz |
| 7 | AT/10/TB IEPE | `vib_x`, `vib_y`, `vib_z` | 100 Hz |
| 8 | Kinematic artefact gen | `euler`, `coriolis` | 100 Hz |
| 9 | Kinematic correction | `a_centripetal`, `euler`, `coriolis` | 100 Hz |


---
## Section 0 - Shared Utilities

Imports and helpers reused by every cell.


In [ ]:
import numpy as np
import pandas as pd
import os, json, datetime

np.random.seed(42)



---
## Section 1 - Datasheet Constants & Calibration Coefficients

Values would be loaded from MCU NVM in production.


In [ ]:
# ADXRS645HDYZ gyroscope (ADI datasheet Rev A)
GYRO_VRATIO_V          = 5.0
GYRO_NULL_V_NOM        = 2.5
GYRO_SENS_MV_PER_DPS   = 1.0
GYRO_A0_NULL_DPS       = 0.0
GYRO_A1_NULL_DPS       = 0.08
GYRO_A2_NULL_DPS       = 0.0008
GYRO_B0_SENS           = 1.0
GYRO_B1_SENS           = -0.000033
GYRO_B2_SENS           = -0.0000020
GYRO_ZERO_OFFSET_DPS   = 0.0
ADC_BITS               = 12
ADC_VREF_GYRO          = 5.0

# ADXL206HDZ inclination accelerometer (ADI datasheet Rev C)
ACCEL_VS_V             = 5.0
ACCEL_SENS_MV_PER_G    = 312.0
ACCEL_ZERO_G_V         = 2.5
ACCEL_C0_X = 0.0;  ACCEL_C1_X =  0.20;  ACCEL_C2_X = 0.001
ACCEL_C0_Y = 0.0;  ACCEL_C1_Y = -0.15;  ACCEL_C2_Y = 0.0008
ACCEL_C0_Z = 0.0;  ACCEL_C1_Z =  0.30;  ACCEL_C2_Z = 0.0012
ACCEL_K_SENS_PPM_PER_C = -20.0
ACCEL_ZERO_G_CAL_X     = 0.0
ACCEL_ZERO_G_CAL_Y     = 0.0
ACCEL_ZERO_G_CAL_Z     = 0.0
ADC_VREF_ACCEL         = 5.0

# AT/10/TB IEPE vibration chain
IEPE_SENS_MV_PER_G           = 10.0
IEPE_SENS_TC_PPM_PER_C       = -500.0
WILSON_ICP_TC_PPM_PER_C      = 176.0
IEPE_TOTAL_SENS_TC_PPM_PER_C = IEPE_SENS_TC_PPM_PER_C + WILSON_ICP_TC_PPM_PER_C
ADC_FS_V_IEPE                = 3.3
ADC_BIAS_V_IEPE              = 1.65

# LM95172 temperature
LM95172_LSB_PER_C    = 128.0
LM95172_RESOLUTION_C = 1.0 / LM95172_LSB_PER_C
THRESH_NORMAL_C      = 150.0
THRESH_WARNING_C     = 185.0
THRESH_ALARM_C       = 190.0

# Reference temperature
T_REF_C = 25.0

# Geometry placeholders - overwritten by Section 2B
r_offset = np.array([0.015, 0.000, 0.005])
R_matrix = np.eye(3)

print("Datasheet constants loaded")


---
## Section 2 - Static Zero Calibration

Drill stationary, vertical, bit-down. Average 5000+ samples and store as offsets.


In [ ]:
def calibrate_zero_rate_offset(static_samples, sensitivity_V_per_dps,
                                adc_bits, adc_vref, null_V_nom,
                                n_min=5000, label="sensor"):
    """Zero-rate offset for the analogue gyroscope."""
    if len(static_samples) < n_min:
        print(f"  WARNING: only {len(static_samples)} samples (recommend {n_min}+)")
    mean_adc   = float(np.mean(static_samples))
    std_adc    = float(np.std(static_samples))
    adc_full   = 2**adc_bits - 1
    mean_V     = mean_adc / adc_full * adc_vref
    delta_V    = mean_V - null_V_nom
    offset_dps = delta_V / sensitivity_V_per_dps
    noise_dps  = (std_adc / adc_full * adc_vref) / sensitivity_V_per_dps
    print(f"{label}: offset = {offset_dps:.4f} dps, noise = {noise_dps:.4f} dps")
    return {"offset_dps": offset_dps, "noise_dps": noise_dps}


def calibrate_zero_g_offset(static_samples_V, sensitivity_V_per_g,
                              zero_g_V_nom, orientation="zero", label="axis"):
    """Zero-g offset per accelerometer axis."""
    expected_map = {"z_down": 1.0, "z_up": -1.0, "zero": 0.0}
    expected_g   = expected_map.get(orientation, 0.0)
    mean_V   = float(np.mean(static_samples_V))
    std_V    = float(np.std(static_samples_V))
    mean_g   = (mean_V - zero_g_V_nom) / sensitivity_V_per_g
    offset_g = mean_g - expected_g
    print(f"{label} ({orientation}): offset = {offset_g*1000:.3f} mg")
    return {"offset_g": offset_g, "noise_g": std_V / sensitivity_V_per_g}


# Run static calibration with synthetic samples
GYRO_SENS_V_PER_DPS_CAL = GYRO_SENS_MV_PER_DPS / 1000.0
null_adc_ideal      = int(GYRO_NULL_V_NOM / ADC_VREF_GYRO * (2**ADC_BITS - 1))
offset_inject_adc   = int(0.08 * GYRO_SENS_V_PER_DPS_CAL / ADC_VREF_GYRO * (2**ADC_BITS - 1))
static_gyro_adc     = (null_adc_ideal + offset_inject_adc
                       + np.random.normal(0, 0.5, 8000)).astype(int)
gyro_cal_result      = calibrate_zero_rate_offset(
    static_gyro_adc, GYRO_SENS_V_PER_DPS_CAL, ADC_BITS,
    ADC_VREF_GYRO, GYRO_NULL_V_NOM, label="ADXRS645HDYZ"
)
GYRO_ZERO_OFFSET_DPS = gyro_cal_result["offset_dps"]

ACCEL_SENS_V_PER_G_CAL = ACCEL_SENS_MV_PER_G / 1000.0
for axis, orient, c1 in [("X","zero",ACCEL_C1_X),("Y","zero",ACCEL_C1_Y),("Z","z_down",ACCEL_C1_Z)]:
    base = ACCEL_ZERO_G_V + (1.0 if orient=="z_down" else 0.0) * ACCEL_SENS_V_PER_G_CAL
    samples_V = base + np.random.normal(0, 0.0003, 5000)
    cal = calibrate_zero_g_offset(samples_V, ACCEL_SENS_V_PER_G_CAL,
                                    ACCEL_ZERO_G_V, orientation=orient, label=axis)
    if axis == "X": ACCEL_ZERO_G_CAL_X = cal["offset_g"]
    if axis == "Y": ACCEL_ZERO_G_CAL_Y = cal["offset_g"]
    if axis == "Z": ACCEL_ZERO_G_CAL_Z = cal["offset_g"]

print("\n Static zero calibration complete")


---
## Section 2B - Geometry Calibration: R Matrix and r Offset


In [ ]:
def calibrate_rotation_matrix_9attitude(sensor_readings, known_refs):
    """9-attitude rotation matrix calibration via SVD."""
    H        = sensor_readings.T @ known_refs
    U, S, Vt = np.linalg.svd(H)
    d        = np.linalg.det(Vt.T @ U.T)
    R        = Vt.T @ np.diag([1, 1, d]) @ U.T
    residual = np.mean([np.linalg.norm(R @ sensor_readings[i] - known_refs[i])
                        for i in range(len(sensor_readings))])
    print(f"R matrix: det = {np.linalg.det(R):.8f}, residual = {residual*1000:.3f} mg")
    return R, residual


def calibrate_r_offset_spin_fixture(omega_z_rad_s, mean_accel_g,
                                     zero_g_cal_x=0.0, zero_g_cal_y=0.0):
    """Offset vector r via spin fixture."""
    g_SI       = 9.80665
    a_corr_g   = mean_accel_g - np.array([zero_g_cal_x, zero_g_cal_y, 0.0])
    a_cent_ms2 = a_corr_g * g_SI
    omega_sq   = omega_z_rad_s**2
    rx = -a_cent_ms2[0] / omega_sq
    ry = -a_cent_ms2[1] / omega_sq
    print(f"r offset: rx = {rx*1000:.2f} mm, ry = {ry*1000:.2f} mm")
    return np.array([rx, ry, 0.0])


# Run geometry calibration with mock data
theta_x = np.radians(1.2);  theta_y = np.radians(-0.8)
R_true  = np.array([[np.cos(theta_y),0,np.sin(theta_y)],[0,1,0],
                    [-np.sin(theta_y),0,np.cos(theta_y)]]) @ np.array(
                   [[1,0,0],[0,np.cos(theta_x),-np.sin(theta_x)],
                    [0,np.sin(theta_x),np.cos(theta_x)]])
refs = np.array([[1,0,0],[-1,0,0],[0,1,0],[0,-1,0],[0,0,1],[0,0,-1],
                 [1/np.sqrt(2),1/np.sqrt(2),0],
                 [1/np.sqrt(2),0,1/np.sqrt(2)],
                 [0,1/np.sqrt(2),1/np.sqrt(2)]], dtype=float)
sensor_reads = np.array([np.linalg.inv(R_true) @ r + np.random.normal(0, 0.001, 3) for r in refs])
R_matrix, _  = calibrate_rotation_matrix_9attitude(sensor_reads, refs)

spin_rpm    = 120.0
omega_spin  = spin_rpm * 2*np.pi / 60.0
r_true_draw = np.array([0.015, 0.000, 0.005])
a_cent_true = -omega_spin**2 * np.array([r_true_draw[0], r_true_draw[1], 0]) / 9.80665
mean_spin_g = a_cent_true + np.random.normal(0, 0.0002, 3)
r_offset    = calibrate_r_offset_spin_fixture(omega_spin, mean_spin_g,
                                                zero_g_cal_x=ACCEL_ZERO_G_CAL_X,
                                                zero_g_cal_y=ACCEL_ZERO_G_CAL_Y)
r_offset[2] = r_true_draw[2]
print("\n Geometry calibration complete")


---
## Section 2C - Temperature Coefficient Calibration


In [ ]:
def calibrate_temperature_coefficient(temps_c, sensor_readings, true_value,
                                          sensor_name="sensor"):
    """Fit reading(T) = true_value * (B0 + B1*dT + B2*dT^2)."""
    dT         = np.array(temps_c) - T_REF_C
    normalised = np.array(sensor_readings) / true_value
    coeffs     = np.polyfit(dT, normalised, 2)
    B2, B1, B0 = coeffs
    rmse       = np.sqrt(np.mean(((B0 + B1*dT + B2*dT**2) - normalised)**2))
    print(f"{sensor_name}: B0={B0:.4f}, B1={B1*1e6:.1f} ppm/C, RMSE={rmse:.6f}")
    return {"B0": B0, "B1": B1, "B2": B2, "rmse": rmse}


# Run temperature coefficient calibration
test_temps = np.array([-20.0, 0.0, 25.0, 60.0, 100.0, 140.0, 175.0])
dT_test    = test_temps - T_REF_C

sim_gyro = 100.0 * (GYRO_B0_SENS + GYRO_B1_SENS*dT_test + GYRO_B2_SENS*dT_test**2)
sim_gyro += np.random.normal(0, 0.05, len(test_temps))
gyro_tc  = calibrate_temperature_coefficient(test_temps, sim_gyro, 100.0, "ADXRS645HDYZ")

sim_iepe = 1.0 * (1.0 + IEPE_TOTAL_SENS_TC_PPM_PER_C*dT_test/1e6)
sim_iepe += np.random.normal(0, 0.0003, len(test_temps))
iepe_tc  = calibrate_temperature_coefficient(test_temps, sim_iepe, 1.0, "AT/10/TB")

print("\n Temperature coefficient calibration complete")


---
## Section 3 - Run Timing & Time Bases

Defines duration and sample counts for 100 Hz and 1 Hz recording frequency


In [ ]:
RUN_ID      = "ICS_FW2_GEN_20260401_120000"
DURATION_S  = 120
FS_HIGH     = 100         # vibration, RPM [Hz]
FS_LOW      = 1           # inclination, temperature [Hz]
N_HIGH      = DURATION_S * FS_HIGH    # 12000
N_LOW       = DURATION_S * FS_LOW     # 120

t_100hz     = np.linspace(0, DURATION_S, N_HIGH)
t_1hz       = np.linspace(0, DURATION_S, N_LOW)

OUT_DIR     = "./ics_gen_output"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"Run ID    : {RUN_ID}")
print(f"Duration  : {DURATION_S} s")
print(f"100 Hz    : {N_HIGH} samples")
print(f"1 Hz      : {N_LOW} samples")
print(f"Output    : {OUT_DIR}/")


---
## Section 4 - LM95172 Temperature Data Generation

**Returns:** `temp_cjc` (cold junction, PCB), `temp_hjc` (hot junction, borehole).
**Rate:** 1 Hz · **Source:** mock data.


In [ ]:
def generate_temperature_data(n_samples, t_ref_c=25.0, dT_max=107.0):
    """Generate LM95172 cold junction + MAX31856 hot junction temperatures.

    Returns:
        temp_cjc: cold junction (PCB) temperature [C], shape (n_samples,)
        temp_hjc: hot junction (borehole) temperature [C], shape (n_samples,)
    """
    # Cold junction rises with downhole heat-soak from 28 C to T_PEAK
    t_peak   = t_ref_c + dT_max
    temp_cjc = np.linspace(28.0, t_peak, n_samples)
    temp_cjc += np.random.normal(0, 0.4, n_samples)

    # Hot junction (borehole) sits ~18 C above PCB
    temp_hjc = temp_cjc + 18.0 + np.random.normal(0, 0.3, n_samples)
    return temp_cjc, temp_hjc


# Generate
temp_cjc, temp_hjc = generate_temperature_data(N_LOW)

# Return values explicitly
print(f"temp_cjc : shape={temp_cjc.shape}  range=[{temp_cjc.min():.1f}, {temp_cjc.max():.1f}] C")
print(f"temp_hjc : shape={temp_hjc.shape}  range=[{temp_hjc.min():.1f}, {temp_hjc.max():.1f}] C")
print(f"\nReturn values: temp_cjc, temp_hjc")


---
## Section 5 - ADXRS645HDYZ Gyroscope Data Generation

Raw rotation including temperature compensation. Null-drift polynomial subtracted, scale-factor polynomial applied.

**Returns:** `omega_dps` (angular rate, °/s), `rpm` (revolutions per minute).
**Rate:** 100 Hz · **Source:** synthetic random data.


In [ ]:
def _build_rpm_schedule(t_arr, drill_rpm=150.0, drill_s=20.0, survey_s=10.0,
                          ramp_s=2.0):

    rpm_target = np.zeros_like(t_arr)
    cycle      = drill_s + 2*ramp_s + survey_s
    for i, t in enumerate(t_arr):
        phase = t % cycle
        if phase < drill_s:
            rpm_target[i] = drill_rpm
        elif phase < drill_s + ramp_s:
            # Ramp down
            rpm_target[i] = drill_rpm * (1 - (phase - drill_s) / ramp_s)
        elif phase < drill_s + ramp_s + survey_s:
            # Survey window - stationary
            rpm_target[i] = 0.0
        else:
            # Ramp up
            rpm_target[i] = drill_rpm * (phase - drill_s - ramp_s - survey_s) / ramp_s
    return rpm_target


def generate_gyro_data(t_arr, temp_arr_1hz, fs_low=1, drill_rpm=150.0,
                       drill_s=20.0, survey_s=10.0, ramp_s=2.0,
                       null_dps_polynomial=(GYRO_A0_NULL_DPS, GYRO_A1_NULL_DPS, GYRO_A2_NULL_DPS),
                       sens_polynomial=(GYRO_B0_SENS, GYRO_B1_SENS, GYRO_B2_SENS)):
    """Generate temperature-compensated gyroscope output with drill/survey cycles.

    Returns:
        omega_dps: angular rate [deg/s], shape (n_high,)
        rpm:       rev/min from |omega|/6, shape (n_high,)
    """
    n = len(t_arr)
    # Upsample temperature to 100 Hz
    temp_hi = np.interp(t_arr, np.linspace(t_arr[0], t_arr[-1], len(temp_arr_1hz)),
                        temp_arr_1hz)
    dT      = temp_hi - T_REF_C

    # Schedule: alternate drilling and survey windows
    rpm_target = _build_rpm_schedule(t_arr, drill_rpm, drill_s, survey_s, ramp_s)
    # True rotation: scheduled RPM in dps + small torsional oscillation when spinning
    osc        = 25.0 * np.sin(2*np.pi*0.5*t_arr) * (rpm_target > 1.0)
    omega_true = rpm_target * 6.0 + osc

    # Apply null drift and scale TC, then immediately remove via compensation
    a0, a1, a2 = null_dps_polynomial
    b0, b1, b2 = sens_polynomial
    null_drift = a0 + a1*dT + a2*dT**2
    scale_fac  = b0 + b1*dT + b2*dT**2
    omega_raw  = omega_true * scale_fac + null_drift + np.random.normal(0, 0.4, n)

    # Compensation: subtract null, divide by scale
    omega_dps  = (omega_raw - null_drift) / scale_fac
    rpm        = np.abs(omega_dps) / 6.0
    return omega_dps, rpm


# Generate
omega_dps, rpm = generate_gyro_data(t_100hz, temp_cjc)

# Return values explicitly
n_total  = len(omega_dps)
n_survey = int(np.sum(np.abs(omega_dps) < 50.0))
print(f"omega_dps : shape={omega_dps.shape}  range=[{omega_dps.min():.1f}, {omega_dps.max():.1f}] dps")
print(f"rpm       : shape={rpm.shape}        mean={rpm.mean():.1f} RPM")
print(f"  drilling samples (|omega|>=50 dps): {n_total - n_survey}/{n_total}")
print(f"  survey   samples (|omega|< 50 dps): {n_survey}/{n_total}")
print(f"\nReturn values: omega_dps, rpm")


---
## Section 6 - ADXL206HDZ Inclination Data Generation

Raw inclination from dual ADXL206HDZ. Suppressed during rotation per technical reference (`|omega_z| < 50 dps`).

**Returns:** `inclination` (degrees from vertical), `inclination_valid` (1 = valid, 0 = suppressed).
**Rate:** 1 Hz · **Source:** synthetic random data.


In [ ]:
def generate_inclination_data(omega_dps_100hz, fs_high=100, true_incl_deg=4.5,
                                noise_deg=0.13):
    """Generate inclination + validity flag from compensated accel readings.

    Returns:
        inclination:       angle from vertical [deg], shape (n_low,)
        inclination_valid: 1 = |omega_z|<50 dps, 0 = suppressed, shape (n_low,)
    """
    n_low = len(omega_dps_100hz) // fs_high
    # Mean omega per second
    omega_1hz = omega_dps_100hz.reshape(n_low, fs_high).mean(axis=1)
    # Inclination noisy around the true value
    inclination = true_incl_deg + np.random.normal(0, noise_deg, n_low)
    # Valid only when tool near-stationary
    inclination_valid = (np.abs(omega_1hz) < 50.0).astype(int)
    return inclination, inclination_valid


# Generate
inclination, inclination_valid = generate_inclination_data(omega_dps)

# Return values explicitly
print(f"inclination       : shape={inclination.shape}  range=[{inclination.min():.2f}, {inclination.max():.2f}] deg")
print(f"inclination_valid : shape={inclination_valid.shape}  valid={int(inclination_valid.sum())}/{len(inclination_valid)}")
print(f"\nReturn values: inclination, inclination_valid")


---
## Section 7 - AT/10/TB IEPE Vibration Data Generation

Raw vibration including temperature compensation (sensitivity + ICP current drift). Centripetal artefact is self-eliminated by IEPE AC coupling.

**Returns:** `vib_x`, `vib_y`, `vib_z` (acceleration, g).
**Rate:** 100 Hz · **Source:** synthetic random data.


In [ ]:
def generate_vibration_data(t_arr, temp_arr_1hz,
                              tc_ppm_per_c=IEPE_TOTAL_SENS_TC_PPM_PER_C):
    """Generate temperature-compensated vibration channels.

    Returns:
        vib_x, vib_y, vib_z: acceleration [g], shape (n_high,) each
    """
    n = len(t_arr)
    # Upsample temperature to 100 Hz
    temp_hi = np.interp(t_arr, np.linspace(t_arr[0], t_arr[-1], len(temp_arr_1hz)),
                        temp_arr_1hz)
    dT      = temp_hi - T_REF_C

    # Drift factor reduces apparent sensitivity at high T
    drift = 1.0 + tc_ppm_per_c * dT / 1e6

    # True signals
    vib_x_true = 0.4 * np.sin(2*np.pi*45*t_arr)        + np.random.normal(0, 0.18, n)
    vib_y_true = 0.3 * np.sin(2*np.pi*45*t_arr + 1.0)  + np.random.normal(0, 0.15, n)
    vib_z_true = 0.6 * np.sin(2*np.pi*30*t_arr)        + np.random.normal(0, 0.22, n)

    # Apply drift, then compensate (divide by drift) - what firmware sees
    vib_x = (vib_x_true * drift) / drift
    vib_y = (vib_y_true * drift) / drift
    vib_z = (vib_z_true * drift) / drift
    return vib_x, vib_y, vib_z


# Generate
vib_x, vib_y, vib_z = generate_vibration_data(t_100hz, temp_cjc)

# Return values explicitly
print(f"vib_x : shape={vib_x.shape}  range=[{vib_x.min():.3f}, {vib_x.max():.3f}] g")
print(f"vib_y : shape={vib_y.shape}  range=[{vib_y.min():.3f}, {vib_y.max():.3f}] g")
print(f"vib_z : shape={vib_z.shape}  range=[{vib_z.min():.3f}, {vib_z.max():.3f}] g")
print(f"\nReturn values: vib_x, vib_y, vib_z")


---
## Section 8 - Kinematic Artefact Generation

Computes kinematic artefacts that contaminate the IEPE vibration signal: Euler (from angular acceleration × r) and Coriolis (from 2 ω × v_axial). Inputs are temperature-compensated angular speed and the spin fixture-measured offset vector r.

**Returns:** `euler` (Nx3 array, g), `coriolis` (Nx3 array, g).
**Rate:** 100 Hz · **Source:** synthetic random data.


In [ ]:
def generate_kinematic_artefacts(omega_dps_100hz, r_vec, t_arr,
                                  rop_m_per_s=0.005, fs_high=100):
    """Compute Euler and Coriolis artefacts from omega and offset r.

    Args:
        omega_dps_100hz : Z-axis angular rate [deg/s], shape (n_high,)
        r_vec           : sensor offset from rotation axis [m], shape (3,)
        t_arr           : 100 Hz time vector
        rop_m_per_s     : axial rate of penetration [m/s]
        fs_high         : sample rate [Hz]

    Returns:
        euler   : Euler artefact accel [g], shape (n_high, 3)
        coriolis: Coriolis artefact accel [g], shape (n_high, 3)
    """
    g_SI       = 9.80665
    omega_rads = np.deg2rad(omega_dps_100hz)
    # Angular acceleration via finite difference + simple low-pass smoothing
    alpha      = np.gradient(omega_rads, 1.0/fs_high)
    alpha      = pd.Series(alpha).rolling(20, min_periods=1, center=True).mean().to_numpy()

    # Euler artefact = alpha x r  (only z-component of alpha)
    euler = np.zeros((len(t_arr), 3))
    euler[:, 0] = -alpha * r_vec[1]
    euler[:, 1] =  alpha * r_vec[0]
    euler[:, 2] =  0.0
    euler /= g_SI

    # Coriolis artefact = 2 omega x v_axial (v_axial = [0, 0, ROP])
    v_axial  = np.array([0.0, 0.0, rop_m_per_s])
    coriolis = np.zeros((len(t_arr), 3))
    coriolis[:, 0] = -2 * omega_rads * v_axial[1]
    coriolis[:, 1] =  2 * omega_rads * v_axial[0]
    coriolis[:, 2] =  0.0
    coriolis /= g_SI
    return euler, coriolis


# Generate
euler, coriolis = generate_kinematic_artefacts(omega_dps, r_offset, t_100hz)

# Return values explicitly
print(f"euler    : shape={euler.shape}     |max|={np.abs(euler).max():.4f} g")
print(f"coriolis : shape={coriolis.shape}  |max|={np.abs(coriolis).max():.6f} g")
print(f"\nReturn values: euler, coriolis")


---
## Section 9 - Kinematic Correction (ADXL206HDZ)

For the ADXL206HDZ inclination accelerometer (DC-coupled, unlike IEPE), the centripetal artefact is the largest contaminant. This cell computes all three artefact terms ready to subtract.

**Returns:** `a_centripetal`, `euler`, `coriolis` (each Nx3 array, g).
**Rate:** 100 Hz · **Source:** synthetic random data.


In [ ]:
def kinematic_correction(omega_dps_100hz, r_vec, t_arr,
                          rop_m_per_s=0.005, fs_high=100):
    """Compute all three kinematic artefacts for the inclination accel.

    Returns:
        a_centripetal: centripetal artefact [g], shape (n_high, 3)
        euler        : Euler artefact [g], shape (n_high, 3)
        coriolis     : Coriolis artefact [g], shape (n_high, 3)
    """
    g_SI       = 9.80665
    omega_rads = np.deg2rad(omega_dps_100hz)

    # Centripetal: a = -omega^2 * r, x and y components
    a_centripetal = np.zeros((len(t_arr), 3))
    a_centripetal[:, 0] = -omega_rads**2 * r_vec[0]
    a_centripetal[:, 1] = -omega_rads**2 * r_vec[1]
    a_centripetal[:, 2] =  0.0
    a_centripetal /= g_SI

    # Reuse Euler + Coriolis from Section 8 helper
    euler, coriolis = generate_kinematic_artefacts(omega_dps_100hz, r_vec, t_arr,
                                                     rop_m_per_s, fs_high)
    return a_centripetal, euler, coriolis


# Generate
a_centripetal, euler_corr, coriolis_corr = kinematic_correction(omega_dps, r_offset, t_100hz)

# Return values explicitly
print(f"a_centripetal : shape={a_centripetal.shape}  |max|={np.abs(a_centripetal).max():.4f} g")
print(f"euler         : shape={euler_corr.shape}     |max|={np.abs(euler_corr).max():.4f} g")
print(f"coriolis      : shape={coriolis_corr.shape}  |max|={np.abs(coriolis_corr).max():.6f} g")
print(f"\nReturn values: a_centripetal, euler, coriolis")


---
## Section 10 - JSON Export

Serialises every return value to a single JSON file. Top-level keys group by section. Each leaf key matches the variable name returned by its generator cell.


In [ ]:
def _ds(arr, n=None):
    """Convert numpy array to plain list, optionally downsampled."""
    a = np.asarray(arr)
    if n is None or len(a) <= n:
        if a.ndim == 1:
            return [round(float(x), 6) for x in a]
        else:
            return [[round(float(v), 6) for v in row] for row in a]
    step = max(1, len(a)//n)
    return _ds(a[::step][:n])


def export_returns_json(filepath, downsample_high=600, downsample_low=120):
    """Write all return values to a JSON file with grouped headings."""

    payload = {
        "meta": {
            "run_id":       RUN_ID,
            "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
            "duration_s":   DURATION_S,
            "fs_high_hz":   FS_HIGH,
            "fs_low_hz":    FS_LOW,
            "n_high":       N_HIGH,
            "n_low":        N_LOW,
            "data_source":  "SYNTHETIC RANDOM DATA - NOT MEASURED FROM HARDWARE",
        },

        "section_4_lm95172_temperature": {
            "rate_hz":  FS_LOW,
            "returns":  ["temp_cjc", "temp_hjc"],
            "temp_cjc": _ds(temp_cjc, downsample_low),
            "temp_hjc": _ds(temp_hjc, downsample_low),
        },

        "section_5_adxrs645_gyroscope": {
            "rate_hz":   FS_HIGH,
            "returns":   ["omega_dps", "rpm"],
            "omega_dps": _ds(omega_dps, downsample_high),
            "rpm":       _ds(rpm,       downsample_high),
        },

        "section_6_adxl206_inclination": {
            "rate_hz":           FS_LOW,
            "returns":           ["inclination", "inclination_valid"],
            "inclination":       _ds(inclination,       downsample_low),
            "inclination_valid": _ds(inclination_valid, downsample_low),
        },

        "section_7_at10tb_vibration": {
            "rate_hz": FS_HIGH,
            "returns": ["vib_x", "vib_y", "vib_z"],
            "vib_x":   _ds(vib_x, downsample_high),
            "vib_y":   _ds(vib_y, downsample_high),
            "vib_z":   _ds(vib_z, downsample_high),
        },

        "section_8_kinematic_artefacts": {
            "rate_hz":  FS_HIGH,
            "returns":  ["euler", "coriolis"],
            "euler":    _ds(euler,    downsample_high),
            "coriolis": _ds(coriolis, downsample_high),
        },

        "section_9_kinematic_correction": {
            "rate_hz":       FS_HIGH,
            "returns":       ["a_centripetal", "euler", "coriolis"],
            "a_centripetal": _ds(a_centripetal, downsample_high),
            "euler":         _ds(euler_corr,    downsample_high),
            "coriolis":      _ds(coriolis_corr, downsample_high),
        },
    }

    with open(filepath, "w") as f:
        json.dump(payload, f, indent=2)

    print(f"JSON exported: {filepath}")
    print(f"  Size: {os.path.getsize(filepath)/1024:.1f} kB")
    print(f"  Sections: {len([k for k in payload if k.startswith('section_')])}")
    return filepath


json_path = os.path.join(OUT_DIR, f"{RUN_ID}_data_returns.json")
export_returns_json(json_path)
